# General Tasks – SoSe26 Case Study, Group 43

Group members: Nicolas Alexander Bauer, Roger Alexander Beever, Tobias Fabian Dünnebeil, Baptist Emil Orb, Louise Charlotte Zepter

**Note on file paths:** All paths in this notebook are *relative*. The notebook expects the original `Data`
folder (as provided via tubCloud) to be placed in the same directory as this notebook, following the
structure `Data/IDA SoSe26 - Data/Komponente/...`. Do **not** hard-code absolute/Windows paths – this keeps
the notebook reproducible on any machine (see Submission Requirements).


## 0. Setup and Imports

In [ ]:
import warnings
warnings.filterwarnings("ignore")  # suppress non-critical warnings for a clean, readable notebook

from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
np.random.seed(42)

# Relative paths to the data folder (place the original "Data" folder next to this notebook).
# Folder names match the official "IDA SoSe26 - Data" structure provided via tubCloud.
BASE_DATA_DIR = Path("Data") / "IDA SoSe26 - Data"

KOMPONENTE_DIR = BASE_DATA_DIR / "Komponente"          # Bestandteile_Komponente_*.csv (BOM/bridge tables)
LOGISTIKVERZUG_DIR = BASE_DATA_DIR / "Logistikverzug"  # Komponente_K7.csv, Logistikverzug_K7.csv
EINZELTEIL_DIR = BASE_DATA_DIR / "Einzelteil"          # Einzelteil_T16.txt
ZULASSUNG_DIR = BASE_DATA_DIR / "Zulassungen"          # Zulassungen_alle_Fahrzeuge.csv
FAHRZEUG_DIR = BASE_DATA_DIR / "Fahrzeug"              # Fahrzeuge_OEM1_Typ11_Fehleranalyse.csv, Bestandteile_Fahrzeuge_*.csv

# Alias used by the "Bestandteile_Komponente_*" bridge-table cells (T16, K1, ...):
DATA_DIR = KOMPONENTE_DIR
BESTANDTEILE_DIR = FAHRZEUG_DIR


## 1. Logistics and Product Development in the Automobile Industry (8 Points)

### 1.0 Objective

We want to characterize the **logistics delay** of component **K7**: the time between when a produced unit
is issued at the supplier (one day after `Produktionsdatum`) and when it is registered as incoming goods at
the OEM (`Wareneingang`). We will:

1. build a combined "Logistics delay" dataset from `Komponente_K7.csv` and `Logistikverzug_K7.csv`,
2. identify a suitable probability distribution for the delay (goodness-of-fit tests),
3. compute the mean delay in **working days**,
4. visualize the distribution with Plotly, and
5. describe how a decision tree could classify defective (`Fehlerhaft`) components.


### 1.1 Data Import

We use two datasets:

| Dataset | Relevant columns | Role |
|---|---|---|
| `Komponente_K7.csv` | `IDNummer`, `Produktionsdatum`, `Fehlerhaft` | production date of each component |
| `Logistikverzug_K7.csv` | `IDNummer`, `Wareneingang` | date the component arrived at the OEM |

The key variable that connects both datasets is **`IDNummer`**, the unique identifier of a K7 component.


In [ ]:
komponente = pd.read_csv(LOGISTIKVERZUG_DIR / "Komponente_K7.csv", sep=";")
logistik = pd.read_csv(LOGISTIKVERZUG_DIR / "Logistikverzug_K7.csv", sep=",")

print("Komponente_K7.csv:", komponente.shape)
print("Logistikverzug_K7.csv:", logistik.shape)
komponente.head()


In [ ]:
logistik.head()


### 1.2 Data Preparation and Validation

Before merging, we check data types, missing values, duplicates, and whether the key variable
(`IDNummer`) actually matches between both tables — this is essential to trust the later merge.


In [ ]:
for name, df in [("Komponente_K7", komponente), ("Logistikverzug_K7", logistik)]:
    print(f"--- {name} ---")
    print(df.dtypes)
    print("Missing values:\n", df.isna().sum())
    print("Duplicate IDNummer:", df["IDNummer"].duplicated().sum())
    print()


In [ ]:
# Convert date columns from string to datetime
komponente["Produktionsdatum"] = pd.to_datetime(komponente["Produktionsdatum"])
logistik["Wareneingang"] = pd.to_datetime(logistik["Wareneingang"])

# Check how many IDNummer values are shared between both tables (merge feasibility)
common_ids = set(komponente["IDNummer"]).intersection(set(logistik["IDNummer"]))
print(f"IDs in Komponente_K7: {komponente['IDNummer'].nunique()}")
print(f"IDs in Logistikverzug_K7: {logistik['IDNummer'].nunique()}")
print(f"IDs present in BOTH tables: {len(common_ids)}")


All `IDNummer` values are unique in both tables and match completely between the two datasets
(306,490 common IDs), so an **inner join on `IDNummer`** is safe and will not create or drop rows unexpectedly.


### 1.3 Creating the "Logistics delay" Dataset

The task states that produced goods are issued **one day after** the production date. We therefore define:

- `Ausgabedatum` (issue date) = `Produktionsdatum` + 1 day
- `Verzug_Kalendertage` (delay in calendar days) = `Wareneingang` − `Ausgabedatum`


In [ ]:
logistics_delay = komponente[["IDNummer", "Produktionsdatum", "Fehlerhaft"]].merge(
    logistik[["IDNummer", "Wareneingang"]], on="IDNummer", how="inner", validate="one_to_one"
)

logistics_delay["Ausgabedatum"] = logistics_delay["Produktionsdatum"] + pd.Timedelta(days=1)
logistics_delay["Verzug_Kalendertage"] = (
    logistics_delay["Wareneingang"] - logistics_delay["Ausgabedatum"]
).dt.days

print("Rows in merged 'Logistics delay' dataset:", len(logistics_delay))
logistics_delay.head()


In [ ]:
# Validate the merge: no missing values, no negative delays (goods cannot arrive before they are issued)
print("Missing values after merge:\n", logistics_delay.isna().sum())
print("\nNegative delays (data quality issue if > 0):", (logistics_delay["Verzug_Kalendertage"] < 0).sum())
print("\nDescriptive statistics of the delay (calendar days):")
logistics_delay["Verzug_Kalendertage"].describe()


The merge is complete (no missing values, `validate="one_to_one"` confirms a clean 1:1 join) and no
negative delays occur, so the derived dataset is analysis-ready.


### 1a. How is the logistics delay distributed? (2 Points)

**Approach:** We first inspect the shape of the empirical distribution (histogram, skewness, kurtosis).
Since the delay is a **strictly positive, right-skewed count-like variable** (goods can never arrive
"too early", but can be considerably late), typical candidate distributions are the **Normal**, **Log-normal**
and **Gamma** distribution. We fit each candidate to the data with maximum-likelihood estimation
(`scipy.stats.<dist>.fit`) and compare them using:

- **Kolmogorov–Smirnov (KS) test** – compares the empirical CDF to the fitted theoretical CDF,
- **AIC / BIC** – penalized log-likelihood, to rank distributions independently of sample size effects,
- a **Q–Q plot** – visual check of the fit in the tails.

**Important caveat:** with a very large sample (n ≈ 306,000) and a *discrete* underlying variable (delay is
measured in whole days), formal hypothesis tests such as KS or Chi² become extremely "powerful" — they will
reject almost *any* continuous null distribution at p ≈ 0, even a good approximation, simply because of the
sample size. We therefore treat the p-value as informative mainly in *relative* terms (comparing test
statistics/AIC across candidates) rather than as a strict pass/fail criterion, and support the decision with
visual diagnostics.


In [ ]:
data = logistics_delay["Verzug_Kalendertage"].values.astype(float)

print(f"Mean:      {data.mean():.3f}")
print(f"Std.dev.:  {data.std():.3f}")
print(f"Skewness:  {stats.skew(data):.3f}   (0 = symmetric, >0 = right-skewed)")
print(f"Kurtosis:  {stats.kurtosis(data):.3f}")


In [ ]:
def aic_bic(dist, params, data):
    """Compute AIC/BIC for a fitted scipy.stats distribution."""
    log_lik = np.sum(dist.logpdf(data, *params))
    k = len(params)
    n = len(data)
    aic = 2 * k - 2 * log_lik
    bic = k * np.log(n) - 2 * log_lik
    return aic, bic

candidates = ["norm", "lognorm", "gamma"]
fit_results = []

for name in candidates:
    dist = getattr(stats, name)
    params = dist.fit(data)
    D, p_value = stats.kstest(data, name, args=params)
    aic, bic = aic_bic(dist, params, data)
    fit_results.append({"distribution": name, "params": params, "KS_D": D, "KS_p": p_value, "AIC": aic, "BIC": bic})

fit_df = pd.DataFrame(fit_results).sort_values("AIC").reset_index(drop=True)
fit_df


In [ ]:
# Q-Q plots for visual comparison of the two best candidates
fig = make_subplots(rows=1, cols=2, subplot_titles=("Q-Q Plot: Normal", "Q-Q Plot: Log-normal"))

for col, dist_name in zip([1, 2], ["norm", "lognorm"]):
    dist = getattr(stats, dist_name)
    params = fit_df.loc[fit_df["distribution"] == dist_name, "params"].values[0]
    osm, osr = stats.probplot(data, dist=dist, sparams=params, fit=False)  # osm, osr are both 1D arrays
    fig.add_trace(go.Scatter(x=osm, y=osr, mode="markers", marker=dict(size=3), name=dist_name), row=1, col=col)
    line = np.linspace(osm.min(), osm.max(), 2)
    fig.add_trace(go.Scatter(x=line, y=line, mode="lines", line=dict(color="red", dash="dash"), showlegend=False), row=1, col=col)

fig.update_layout(height=450, width=950, title_text="Q-Q Plots: Theoretical vs. Empirical Quantiles")
fig.show()


**Interpretation:** The delay distribution is **right-skewed** (skewness ≈ 0.57), i.e. most components
arrive within a fairly narrow band around the mean, but a longer tail of late deliveries pulls the
distribution to the right. Consistent with this, the **Log-normal** and **Gamma** distributions achieve a
clearly lower AIC/BIC than the Normal distribution, and their Q-Q plot follows the reference line more
closely in the upper tail. All KS tests reject exact equality (p ≈ 0) — expected given the large,
discrete sample discussed above — but based on the **relative fit (AIC/BIC) and the Q-Q diagnostics**, we
conclude that the logistics delay is best approximated by a **Log-normal distribution** (a Gamma
distribution is a close second and would also be a defensible choice). A Normal distribution is not a good
choice because it does not respect the strictly positive support and underestimates the right tail.


### 1b. Mean logistics delay in working days (2 Points)

We now express the delay in **working days** (Monday–Friday only, weekends excluded), using
`numpy.busday_count`, which counts business days between the issue date (`Ausgabedatum`, inclusive) and the
arrival date (`Wareneingang`, exclusive).


In [ ]:
start_dates = logistics_delay["Ausgabedatum"].values.astype("datetime64[D]")
end_dates = logistics_delay["Wareneingang"].values.astype("datetime64[D]")

logistics_delay["Verzug_Arbeitstage"] = np.busday_count(start_dates, end_dates)

mean_working_days = logistics_delay["Verzug_Arbeitstage"].mean()
median_working_days = logistics_delay["Verzug_Arbeitstage"].median()

print(f"Mean logistics delay:   {mean_working_days:.2f} working days")
print(f"Median logistics delay: {median_working_days:.2f} working days")
print(f"Std. dev.:              {logistics_delay['Verzug_Arbeitstage'].std():.2f} working days")


**Interpretation:** On average, a K7 component takes about **4.3 working days** to travel from the
supplier to the OEM's incoming-goods department (compared to ≈ 6.1 *calendar* days — the difference is
explained by the weekend days that are excluded from the working-day count).

**Alternatives to the arithmetic mean:**
- The **median** is more robust to the right-skewed tail of very late deliveries and may better represent
  the "typical" delay experienced by most shipments.
- A **trimmed mean** (e.g., excluding the top/bottom 1–5 %) would reduce the influence of extreme outliers
  while still using most of the data.
- Reporting the mean **together with a percentile-based service level** (e.g., "90 % of shipments arrive
  within X working days") is often more actionable for logistics planning than a single average, because it
  directly informs buffer/safety-stock decisions.
- Public holidays are not accounted for by `numpy.busday_count` by default; including a holiday calendar
  (`numpy.busdaycalendar`) would make the working-day estimate more precise.


### 1c. Visualization: Histogram and Density Function (2 Points)

We visualize the calendar-day delay distribution with **Plotly**, overlaying the empirical histogram with
the fitted Log-normal density curve identified in 1a.

**Bin size selection:** Since the delay is an *integer-valued* variable (whole days), the most interpretable
choice is **one bin per integer day** (bin width = 1), so that each bar directly represents "share of
shipments with exactly k days of delay" rather than mixing several day-values into one bar. This also avoids
arbitrary binning rules (e.g., Freedman–Diaconis, Sturges) producing bin edges that fall *between* integers,
which would be misleading for discrete data.


In [ ]:
best_dist_name = fit_df.iloc[0]["distribution"]  # best fit by AIC (lognorm)
best_params = fit_df.iloc[0]["params"]
best_dist = getattr(stats, best_dist_name)

x_range = np.linspace(data.min(), data.max(), 300)
pdf_values = best_dist.pdf(x_range, *best_params)

bin_edges = np.arange(data.min() - 0.5, data.max() + 1.5, 1)  # one bin per integer day

fig = go.Figure()
fig.add_trace(go.Histogram(
    x=data, xbins=dict(start=bin_edges[0], end=bin_edges[-1], size=1),
    histnorm="probability density", name="Empirical delay", marker_color="#4C72B0", opacity=0.75
))
fig.add_trace(go.Scatter(
    x=x_range, y=pdf_values, mode="lines", name=f"Fitted {best_dist_name} density",
    line=dict(color="#C44E52", width=3)
))

fig.update_layout(
    title="Distribution of the K7 Logistics Delay (Calendar Days)",
    xaxis_title="Delay [calendar days]",
    yaxis_title="Density",
    bargap=0.05,
    template="plotly_white",
    width=850, height=500
)
fig.show()


### 1d. Decision Tree for Classifying Defective Components (2 Points)

**Task:** describe the process of building a decision tree that classifies whether a K7 component is
defective (`Fehlerhaft`) or not.

**Process:**

1. **Define features and target.** The target is the binary column `Fehlerhaft` (0 = OK, 1 = defective).
   Candidate features available in `Komponente_K7.csv` are `Herstellernummer`, `Werksnummer`, and
   time-derived features from `Produktionsdatum` (e.g., production year/month/weekday) — the reasoning being
   that certain plants, manufacturers, or production periods may be systematically associated with quality
   issues.
2. **Exploratory check of class balance.** Before modelling, inspect how many defective vs. non-defective
   parts exist — this determines whether special handling (class weights, resampling) is required.
3. **Feature engineering.** Extract numeric/categorical features from the date (year, month, day-of-week),
   and encode categorical variables (`Herstellernummer`, `Werksnummer`) if needed (they are already numeric
   codes here).
4. **Train/test split.** Split the data (e.g. 70/30 or 80/20) using **stratified sampling** on `Fehlerhaft`
   so that both sets preserve the (likely very low) share of defective parts.
5. **Model training.** Fit a `sklearn.tree.DecisionTreeClassifier`, using `class_weight="balanced"` to
   compensate for class imbalance, and constrain complexity (`max_depth`, `min_samples_leaf`) to avoid
   overfitting on a rare-event target.
6. **Visualization & interpretation.** Plot the tree (`plot_tree`) and feature importances to understand
   which splits (e.g., specific plant or production month) are associated with higher defect risk.
7. **Evaluation.** Because the target is rare, accuracy is a misleading metric — use **precision, recall,
   F1-score, and the confusion matrix** (or ROC-AUC) instead, focusing on recall for the defective class if
   the business goal is to catch as many defective parts as possible.

Below is an illustrative implementation with the available features:


In [ ]:
model_data = komponente.copy()
model_data["Produktionsjahr"] = model_data["Produktionsdatum"].dt.year
model_data["Produktionsmonat"] = model_data["Produktionsdatum"].dt.month
model_data["Produktionswochentag"] = model_data["Produktionsdatum"].dt.dayofweek

print("Class balance (Fehlerhaft):")
print(model_data["Fehlerhaft"].value_counts(normalize=True))


In [ ]:
features = ["Herstellernummer", "Werksnummer", "Produktionsjahr", "Produktionsmonat", "Produktionswochentag"]
X = model_data[features]
y = model_data["Fehlerhaft"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

tree_clf = DecisionTreeClassifier(
    max_depth=4, min_samples_leaf=50, class_weight="balanced", random_state=42
)
tree_clf.fit(X_train, y_train)

plt.figure(figsize=(16, 8))
plot_tree(tree_clf, feature_names=features, class_names=["OK", "Fehlerhaft"], filled=True, fontsize=8)
plt.title("Decision Tree: Classifying Defective K7 Components")
plt.show()


In [ ]:
importances = pd.Series(tree_clf.feature_importances_, index=features).sort_values(ascending=False)

fig = px.bar(
    importances, orientation="h", labels={"index": "Feature", "value": "Importance"},
    title="Feature Importance – Decision Tree for 'Fehlerhaft'", template="plotly_white"
)
fig.update_layout(showlegend=False, width=750, height=400)
fig.show()


**Note on this specific dataset:** in this sample, only 6 out of 306,490 components are marked as
defective (≈ 0.002 %). With such extreme class imbalance and only a handful of positive cases, *any*
classifier — however well built — will have very limited statistical power, and results should be
interpreted with caution (the tree above is illustrative of the *process*, not a reliable production model).
In practice, more positive examples and richer features (e.g., material batch, supplier quality scores)
would be needed for a robust classifier.


## 2. Data Storage in Separate Files (2 Points)

**Why store data in separate files instead of one large table?**

1. **Reduced redundancy** – shared attributes (e.g., `Herstellernummer`, `Werksnummer`) are stored once per
   entity instead of being repeated in every row of a giant combined table, saving storage and avoiding
   update anomalies.
2. **Data integrity & consistency** – if a value (e.g., a plant's address) needs to change, it only has to be
   updated in one place, instead of in every row of a flattened mega-table, preventing inconsistent copies.
3. **Modularity and maintainability** – each file/table represents one entity (components, logistics events,
   registrations, etc.), which makes the schema easier to understand, extend, and maintain than a single wide
   table mixing unrelated concepts.
4. **Performance and scalability** – smaller, well-indexed tables are faster to query, join selectively, and
   load into memory only when needed, whereas one huge denormalized table wastes memory and I/O on columns
   that are irrelevant to a given analysis.
5. **Referential integrity** – keys (e.g., `IDNummer`) can enforce valid relationships between tables (e.g.,
   a `Bestandteile_Komponente_K7` record must reference an existing component), which is not possible within
   a single flat file.

**Structure name:** The provided tables (separate entity tables connected via key/ID columns, e.g.
`IDNummer`, `ID_T34`, …) follow the structure of a **relational database** (normalized, entity–relationship
model). The specific pattern here — a central "bill of materials"-style bridge table (`Bestandteile_...`)
linking a component to its constituent parts — is a classic **relational (normalized) schema** used to model
hierarchical part structures.


## 3. Parts T16 in Registered Vehicles (3 Points)

**Task:** Determine how many parts T16 ended up in vehicles registered in Adelshofen.

**Approach:** Answering this requires tracing a chain of keys across four data sources:

$$\text{T16 part} \;\rightarrow\; \text{seat sub-type (Sitze)} \;\rightarrow\; \text{vehicle (via the vehicle BOM)} \;\rightarrow\; \text{registration (Zulassung)}$$

We use:
- `Einzelteil_T16.txt` — production/failure data of individual T16 parts,
- `Bestandteile_Komponente_K2LE2.csv` and `Bestandteile_Komponente_K2ST2.csv` — link T16 parts to the two
  seat sub-types that are built from a T16 part (`Bestandteile_Komponente_K2ST1.csv` and
  `..._K2LE1.csv` were checked and confirmed to **not** contain a `T16` column, so only these two matter),
- `Bestandteile_Fahrzeuge_OEM{1,2}_Typ{11,12,21,22}.csv` — link each vehicle to its seat component,
- `Zulassungen_alle_Fahrzeuge.csv` — vehicle registrations.


### 3.1 Data Cleaning: `Einzelteil_T16.txt`

The raw export uses a non-standard structure: there are no line breaks (rows are tab-separated) and fields
are separated by `" | | "`. In addition, the header lists the same 7 columns three times with suffixes
`.x`, `.y`, and unsuffixed — a leftover artifact from a prior column-wise merge in which three
production batches were combined side by side instead of stacked. Each row has exactly one of the three
blocks populated (the other two are `NA`).

We parse the file with the correct delimiters and then stack the three blocks into one clean, long table of
individual T16 parts.


In [ ]:
import io

with open(EINZELTEIL_DIR / "Einzelteil_T16.txt", "r", encoding="utf-8", errors="replace") as f:
    raw_content = f.read()

# Replace the non-standard delimiters with a proper CSV structure:
# " | | " -> field separator ";", tab "\t" -> row separator "\n"
cleaned_content = raw_content.replace(" | | ", ";").replace("\t", "\n")

# The header lists 22 names but each row has 23 fields (an unnamed row-index column precedes
# the three .x / .y / unsuffixed column groups) -> supply column names explicitly
base_cols = ["ID_T16", "Produktionsdatum", "Herstellernummer", "Werksnummer",
             "Fehlerhaft", "Fehlerhaft_Datum", "Fehlerhaft_Fahrleistung"]
col_names = ["_rowname", "_rowidx"] + [c + "_x" for c in base_cols] + [c + "_y" for c in base_cols] + base_cols

einzelteil_raw = pd.read_csv(
    io.StringIO(cleaned_content), sep=";", header=None, names=col_names,
    skiprows=1, na_values="NA", quotechar='"', low_memory=False
)

# Stack the three column-blocks (.x / .y / unsuffixed) into one long table
blocks = []
for suffix in ["_x", "_y", ""]:
    block = einzelteil_raw[[c + suffix for c in base_cols]].copy()
    block.columns = base_cols
    blocks.append(block)

einzelteil_t16 = pd.concat(blocks, ignore_index=True).dropna(subset=["ID_T16"]).reset_index(drop=True)
einzelteil_t16["Produktionsdatum"] = pd.to_datetime(einzelteil_t16["Produktionsdatum"])

print("Cleaned 'Einzelteil_T16' shape:", einzelteil_t16.shape)
print("Duplicate ID_T16 after cleaning:", einzelteil_t16["ID_T16"].duplicated().sum())
einzelteil_t16.head()


### 3.2 Vehicles Registered in Adelshofen

The `Gemeinden` column contains two spellings for the same location, `"ADELSHOFEN"` and `"ADELSHOFEN1"`
(likely a naming-collision artifact from combining regional registration files); we treat both as Adelshofen.


In [ ]:
zulassungen = pd.read_csv(ZULASSUNG_DIR / "Zulassungen_alle_Fahrzeuge.csv", sep=";")
zulassungen["Zulassung"] = pd.to_datetime(zulassungen["Zulassung"])

vehicles_adelshofen = zulassungen[zulassungen["Gemeinden"].isin(["ADELSHOFEN", "ADELSHOFEN1"])]
print(f"Vehicles registered in Adelshofen: {len(vehicles_adelshofen)}")


### 3.3 Linking Vehicles to Their Seat Component

Each vehicle's components (Karosserie, Schaltung, Sitze, Motor) are listed in one BOM file per OEM/vehicle
type. We combine all four, then keep only the `ID_Sitze` (seat) column, join with the Adelshofen vehicles,
and finally join with the T16 bridge tables for the two seat sub-types that contain a T16 part.


In [ ]:
bom_files = {
    "OEM1_Typ11": "Bestandteile_Fahrzeuge_OEM1_Typ11.csv",
    "OEM1_Typ12": "Bestandteile_Fahrzeuge_OEM1_Typ12.csv",
    "OEM2_Typ21": "Bestandteile_Fahrzeuge_OEM2_Typ21.csv",
    "OEM2_Typ22": "Bestandteile_Fahrzeuge_OEM2_Typ22.csv",
}

bom_parts = []
for label, filename in bom_files.items():
    part = pd.read_csv(BESTANDTEILE_DIR / filename, sep=";")
    part = part.drop(columns=[c for c in part.columns if c.startswith("Unnamed") or c == "X1"])
    part["OEM_Typ"] = label
    bom_parts.append(part)

fahrzeuge_bom = pd.concat(bom_parts, ignore_index=True)

# Validate: every vehicle has exactly one BOM record and vice versa
merge_check = fahrzeuge_bom.merge(zulassungen, left_on="ID_Fahrzeug", right_on="IDNummer", how="outer", indicator=True)
assert (merge_check["_merge"] == "both").all(), "Vehicle BOM and registration data do not match 1:1!"

bom_adelshofen = fahrzeuge_bom.merge(vehicles_adelshofen, left_on="ID_Fahrzeug", right_on="IDNummer")
print(f"Adelshofen vehicles with known seat component: {len(bom_adelshofen)}")
print(bom_adelshofen["ID_Sitze"].str.split("-").str[0].value_counts())


In [ ]:
t16_le2 = pd.read_csv(DATA_DIR / "Bestandteile_Komponente_K2LE2.csv", sep=";")[["ID_T16", "ID_K2LE2"]]
t16_le2 = t16_le2.rename(columns={"ID_K2LE2": "ID_Sitze"})

t16_st2 = pd.read_csv(DATA_DIR / "Bestandteile_Komponente_K2ST2.csv", sep=";")[["ID_T16", "ID_K2ST2"]]
t16_st2 = t16_st2.rename(columns={"ID_K2ST2": "ID_Sitze"})

t16_bridge = pd.concat([t16_le2, t16_st2], ignore_index=True)

# Consistency check: the two bridge tables together should cover every individual T16 part
assert t16_bridge["ID_T16"].nunique() == einzelteil_t16["ID_T16"].nunique(), "T16 bridge tables do not cover all parts!"

t16_adelshofen = t16_bridge.merge(bom_adelshofen, on="ID_Sitze")
n_t16_adelshofen = t16_adelshofen["ID_T16"].nunique()

print(f"T16 parts installed in vehicles registered in Adelshofen: {n_t16_adelshofen}")
print(t16_adelshofen["ID_Sitze"].str.split("-").str[0].value_counts())


### Result

**96 T16 parts** ended up in vehicles registered in Adelshofen: 71 via `K2ST2`-type seats and 25 via
`K2LE2`-type seats, out of 264 Adelshofen-registered vehicles in total (the other seat sub-types present in
Adelshofen, `K2ST1` and `K2LE1`, do not use a T16 part).


## 4. Attributes of the Registration Table (2 Points)

We inspect the data types of `Zulassungen_alle_Fahrzeuge` (already loaded above as `zulassungen`) and classify
each attribute both by its **technical (pandas) data type** and its **statistical measurement scale**.


In [ ]:
print(zulassungen.dtypes)
print()
print("Missing values:\n", zulassungen.isna().sum())
print()
print(f"Unique IDNummer: {zulassungen['IDNummer'].nunique()} (of {len(zulassungen)} rows)")
print(f"Unique Gemeinden: {zulassungen['Gemeinden'].nunique()}")
print(f"Zulassung date range: {zulassungen['Zulassung'].min().date()} to {zulassungen['Zulassung'].max().date()}")


| Attribute | Pandas dtype | Statistical scale | Description |
|---|---|---|---|
| `Unnamed: 0` | `int64` | Discrete numeric (technical) | Row index carried over from the original export; not a substantive attribute of the vehicle. |
| `IDNummer` | `object` (string) | **Nominal** | Unique identifier of a vehicle (e.g. `"11-1-11-1"`). Values are unordered labels; arithmetic on them is meaningless. |
| `Gemeinden` | `object` (string) | **Nominal (categorical)** | Name of the municipality where the vehicle is registered. Unordered categories; contains 5,764 distinct values (including a data-quality duplicate, `"ADELSHOFEN"` vs. `"ADELSHOFEN1"`, see Section 3.3). |
| `Zulassung` | `object` → converted to `datetime64` | **Interval scale (date/time)** | Registration date. Differences between dates are meaningful (e.g. "3 days apart"), but there is no true zero, so ratios are not meaningful. |

**Characteristics of the data types:**
- **Nominal** variables (`IDNummer`, `Gemeinden`) represent unordered categories/labels — the only valid
  operations are equality checks and counting frequencies; no meaningful order or arithmetic exists.
- **Interval-scaled date/time** variables (`Zulassung`) support ordering and computing differences (durations),
  but ratios are not meaningful (e.g. "2018 is not twice 2009").
- Storing `IDNummer` and `Gemeinden` as strings (`object`/pandas `str`) rather than numeric types correctly
  reflects that they are identifiers/labels, not quantities to be averaged or summed.


## 5. Linear Model for Mileage (5 Points)

**Task:** create a linear model from `Fahrzeuge_OEM1_Typ11_Fehleranalyse` relating mileage
(`Fehlerhaft_Fahrleistung`) to suitable variables, and derive recommendations for OEM1.

We follow the model-building process from the lecture (*Build a Model*): **(1) choose a model, (2) test the
model, (3) adjust the model, (4) test the adjusted model** — repeating steps 3–4 until we reach a
satisfactory fit — and implement the final model with `statsmodels` (`sm.add_constant()` + `sm.OLS(y, X).fit()`
+ `.summary()`), as shown in the lecture.


In [ ]:
import statsmodels.api as sm

fehleranalyse = pd.read_csv(FAHRZEUG_DIR / "Fahrzeuge_OEM1_Typ11_Fehleranalyse.csv", sep=",")
fehleranalyse["Fehlerhaft_Datum"] = pd.to_datetime(fehleranalyse["Fehlerhaft_Datum"])

print(fehleranalyse.shape)
print(fehleranalyse.dtypes)
fehleranalyse.head()


### 5.1 Choose a Model

**Visual inspection:** we plot mileage against the two continuous candidate variables to check for a
linear relationship.


In [ ]:
fig = px.scatter(
    fehleranalyse.sample(5000, random_state=42), x="fuel", y="Fehlerhaft_Fahrleistung", color="engine",
    labels={"fuel": "Fuel consumption [l/100km]", "Fehlerhaft_Fahrleistung": "Mileage at failure [km]"},
    title="Mileage at Failure vs. Fuel Consumption, by Engine Class (5,000-point sample)",
    template="plotly_white", opacity=0.5
)
fig.update_layout(width=800, height=500)
fig.show()


**Statistical test:** as in the lecture, we use the **Pearson correlation coefficient test** to check
for a linear relationship between each candidate variable and mileage before including it in the model.


In [ ]:
for column in ["days", "fuel"]:
    r, p = stats.pearsonr(fehleranalyse[column], fehleranalyse["Fehlerhaft_Fahrleistung"])
    print(f"Pearson test '{column}' vs. Fehlerhaft_Fahrleistung: r = {r:.4f}, p = {p:.4g}")


**Interpretation:** `fuel` shows a strong, highly significant positive linear relationship with mileage
(r ≈ 0.69, p ≈ 0) — vehicles with higher fuel consumption travel substantially further before a defect
occurs. `days` shows essentially **no** linear relationship (r ≈ 0.002, p = 0.45) and is therefore not a
useful predictor on its own. `engine` (categorical) is included as a second variable, since the scatter plot
shows visibly different mileage levels by engine class.

**Model choice:** we start with the simplest possible model — the constant mean, as in the lecture — and
then progressively adjust it by adding `fuel` and `engine`.


### 5.2 Test the (Baseline) Model

The simplest model is the constant model $\hat{y} = a_0 = \bar{y}$. We evaluate it with the **RMSE**, the
error metric introduced in the lecture, and inspect its residuals.


In [ ]:
y = fehleranalyse["Fehlerhaft_Fahrleistung"]

baseline_pred = np.full_like(y, fill_value=y.mean(), dtype=float)
rmse_baseline = np.sqrt(np.mean((y - baseline_pred) ** 2))
print(f"Baseline model (mean only): RMSE = {rmse_baseline:,.1f} km")


### 5.3 Adjust the Model: Add `fuel`

We adjust the model by adding `fuel` as a predictor, fit it with `statsmodels` using `sm.add_constant()` +
`sm.OLS(y, X).fit()` (as shown in the lecture), and test it again with RMSE and a residual plot.


In [ ]:
X1 = sm.add_constant(fehleranalyse[["fuel"]])
model_1 = sm.OLS(y, X1).fit()
print(model_1.summary())


In [ ]:
rmse_model1 = np.sqrt(model_1.mse_resid)
print(f"Model 1 (fuel only): RMSE = {rmse_model1:,.1f} km  (baseline was {rmse_baseline:,.1f} km)")

fig = go.Figure()
sample_idx = np.random.choice(len(model_1.resid), 5000, replace=False)
fig.add_trace(go.Scatter(x=model_1.fittedvalues.iloc[sample_idx], y=model_1.resid.iloc[sample_idx],
                          mode="markers", marker=dict(size=3, opacity=0.4)))
fig.add_hline(y=0, line_dash="dash", line_color="red")
fig.update_layout(title="Model 1 (fuel only): Residuals vs. Fitted Values", xaxis_title="Fitted mileage [km]",
                   yaxis_title="Residual [km]", template="plotly_white", width=800, height=450, showlegend=False)
fig.show()


**Test result:** adding `fuel` sharply reduces the RMSE compared to the baseline (mean-only) model — a
clear improvement. However, the residuals still show a **fan-out / systematic spread** rather than scattering
purely randomly around 0, suggesting that at least one more variable is needed.


### 5.4 Adjust the Model Again: Add `engine`

We adjust the model further by adding `engine` (categorical, one-hot encoded with `small` as the reference
category) alongside `fuel`, and test the adjusted model the same way.


In [ ]:
engine_dummies = pd.get_dummies(fehleranalyse["engine"], prefix="engine", drop_first=True).astype(float)
X2 = sm.add_constant(pd.concat([fehleranalyse[["fuel"]], engine_dummies], axis=1))

model_2 = sm.OLS(y, X2).fit()
print(model_2.summary())


In [ ]:
rmse_model2 = np.sqrt(model_2.mse_resid)
print(f"Model 2 (fuel + engine): RMSE = {rmse_model2:,.1f} km")
print(f"Model 1 (fuel only):     RMSE = {rmse_model1:,.1f} km")
print(f"Baseline (mean only):    RMSE = {rmse_baseline:,.1f} km")

fig = go.Figure()
sample_idx = np.random.choice(len(model_2.resid), 5000, replace=False)
fig.add_trace(go.Scatter(x=model_2.fittedvalues.iloc[sample_idx], y=model_2.resid.iloc[sample_idx],
                          mode="markers", marker=dict(size=3, opacity=0.4)))
fig.add_hline(y=0, line_dash="dash", line_color="red")
fig.update_layout(title="Model 2 (fuel + engine): Residuals vs. Fitted Values", xaxis_title="Fitted mileage [km]",
                   yaxis_title="Residual [km]", template="plotly_white", width=800, height=450, showlegend=False)
fig.show()


**Test result:** Model 2 further reduces the RMSE relative to Model 1, and its residuals scatter more evenly
around 0 with a less pronounced pattern. We stop the iterative adjustment here, since `days` was already
ruled out in Section 5.1 (no linear relationship) and no further variables are available in this table.
Model 2 (`fuel` + `engine`) is therefore our final model.


### 5.5 Interpretation and Recommendations for OEM1

**Model fit:** the final model (`fuel` + `engine`) achieves a clearly lower RMSE than both the baseline and
the `fuel`-only model, and its R² (see `model_2.summary()` above) shows it explains a substantial share of
the variance in mileage at failure.

**Key findings:**
1. **Fuel consumption is the strongest driver of mileage at failure** (confirmed by the Pearson test and the
   `fuel` coefficient): every additional liter/100km is associated with several thousand additional
   kilometers driven before a defect occurs.
2. **Engine class adds further explanatory power** beyond fuel consumption alone (visible in the RMSE drop
   from Model 1 to Model 2).
3. **Elapsed time (`days`) has no linear relationship with mileage** (Pearson test, Section 5.1) — failures
   are not simply a function of calendar time, but of usage intensity.

**Recommendations for OEM1:**
- **Prefer mileage-based over purely time-based maintenance/warranty triggers.** Since `days` shows no
  relationship with mileage, a fixed calendar-time policy will systematically under-serve high-mileage
  vehicles and over-serve low-mileage ones.
- **Differentiate reliability targets and inspection intervals by engine class and fuel-consumption level**,
  since these two factors jointly explain most of the variance the model captures.
- **Use fuel consumption as a low-cost, readily available proxy for usage intensity** in predictive
  maintenance planning.
- Since the model does not explain all of the variance, OEM1 should consider collecting additional variables
  (e.g. driving style, road conditions, maintenance history) to improve predictions further.


## 6. Hit and Run Accident Investigation (5 Points)

**Task:** On 11.08.2010, a hit-and-run accident occurred. The vehicle's license plate is unknown, but the
**body part number (Karosserie ID) `K5-112-1122-79`** was recovered. We need to trace which vehicle this
body belongs to and find out where it was registered.

**Approach:**
1. Find `K5-112-1122-79` in the `ID_Karosserie` column of the vehicle BOM table (`fahrzeuge_bom`) — the
   `K5` prefix already tells us it belongs to the **OEM1 Typ12** BOM file.
2. Retrieve the corresponding `ID_Fahrzeug`.
3. Look up that vehicle in `Zulassungen_alle_Fahrzeuge` to find its registration location and date.
4. Sanity-check that the registration date is before the accident date (11.08.2010), i.e. the vehicle was
   already on the road at the time of the accident.


In [ ]:
target_karosserie = "K5-112-1122-79"

match = fahrzeuge_bom[fahrzeuge_bom["ID_Karosserie"] == target_karosserie]
print("BOM record for the recovered body part:")
print(match)

vehicle_id = match["ID_Fahrzeug"].iloc[0]
print(f"\nAssociated vehicle ID: {vehicle_id}")


In [ ]:
registration = zulassungen[zulassungen["IDNummer"] == vehicle_id]
print("Registration record:")
print(registration)

accident_date = pd.Timestamp("2010-08-11")
reg_date = registration["Zulassung"].iloc[0]
print(f"\nRegistration date: {reg_date.date()}  |  Accident date: {accident_date.date()}")
print("Registered before the accident:", reg_date < accident_date)


### Result

The body part `K5-112-1122-79` belongs to vehicle **`12-1-12-82`**, which was registered in
**Aschersleben** on **2009-01-02** — well before the accident date (11.08.2010), so the registration is
consistent with this vehicle being on the road at the time of the hit-and-run. The Federal Motor Transport
Authority can direct the police to the owner of record for vehicle `12-1-12-82` in Aschersleben.
